In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
#pip install wandb 

In [ ]:
'''
import wandb

wandb.init(
    project="22f3000241-t22026"
    
)
'''

In [ ]:
import matplotlib.pyplot as plt


In [ ]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
sample_sub = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [ ]:
train_df.head()

In [ ]:
print(train_df.shape)
print(test_df.shape)

In [ ]:
print(f'train columns: {train_df.columns}')
print(f'test columns: {test_df.columns}')

In [ ]:
train_df.isnull().sum()

In [ ]:
test_df.isnull().sum()

# Answer Label Distribution

In [ ]:
train_df['answer'].describe()

In [ ]:
answer_count = train_df['answer'].value_counts()
answer_count

In [ ]:
percentages = (train_df['answer'].value_counts()/len(train_df))*100
percentages


In [ ]:
ax = answer_count.plot(kind='bar')
for i, p in enumerate(percentages):
    ax.text(i, answer_count.iloc[i] + 5, f'{p:.1f}%', ha='center')
plt.xlabel('Answer')
plt.ylabel('Count')
plt.title('Distribution of Answers')
plt.show()

# Understanding Prompts

In [ ]:
train_df['prompt'].str.len()

In [ ]:
train_df['prompt'].describe()

In [ ]:
print(f'Total number of duplicates present in prompt : {train_df['prompt'].duplicated().sum()}')

In [ ]:
print(f'Total number of unique prompts : {train_df['prompt'].nunique()}')

In [ ]:
train_df[train_df['prompt'].duplicated()].head()

# OPTIONS

In [ ]:
len(train_df)

In [ ]:
train_df['id']

In [ ]:
options=['A','B','C','D']

In [ ]:
options=['A','B','C','D']

avg_lengths = []

for option in options:
    avg_lengths.append(train_df[option].str.len().mean())

print(avg_lengths)
    

In [ ]:

plt.bar(options, avg_lengths)
plt.xlabel("Option")
plt.ylabel("Average Length")
plt.title("Average Length of Each Option")
plt.show()

In [ ]:
train_df['correct_len'] = train_df.apply(
    lambda row: len(row[row['answer']]),
    axis=1
)
correct_len = train_df['correct_len'].mean()
correct_len
print(f'Average of correct answer length: {correct_len:.2f}')

In [ ]:
avg_correct = train_df['correct_len'].mean()

plt.bar(options + ['Correct'], avg_lengths + [avg_correct])
plt.xlabel('Option')
plt.ylabel('Average Length')
plt.title('Average Length of Options vs Correct Answers')
plt.show()

# Test Data

In [ ]:
train_prompt=set(train_df['prompt'])


In [ ]:
test_df['in_train'] = test_df['prompt'].isin(train_prompt)
print(f'No of questions present in train dataset which is also present in the test dataset : {test_df['in_train'].sum()} out of {len(test_df)}') 
print(f'In Percentage : {test_df['in_train'].mean()*100}%')



In [ ]:
lookup = dict(zip(train_df['prompt'], train_df['answer']))
test_df['known_answer'] = test_df['prompt'].map(lookup)


In [ ]:
print(test_df[['id', 'prompt', 'known_answer']].dropna())

# Baseline

In [ ]:
test_df.columns

In [ ]:
test_df.head()

In [ ]:
def make_pred(known_answer):
    freq_order = ['B', 'C', 'A', 'D', 'E']
    
    # Step 2: remove known_answer from the list
    remaining = [x for x in freq_order if x != known_answer ]
    
    # Step 3: take first 2
    top2 = remaining[:2]
    
    # Step 4: combine known_answer + top2, join with space
    pred = known_answer + " " + top2[0] + " " + top2[1]
    
    return pred

In [ ]:
predictions = []
for i, row in test_df.iterrows():
    if pd.notna(row['known_answer']):
        pred = make_pred(row['known_answer'])
    else:
        pred = "B C A"
    predictions.append(pred)

In [ ]:
len(predictions)

In [ ]:
sub=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
sub

In [ ]:
'''
submission = pd.DataFrame({
    "ID":test_df['id'],
    "Prediction":predictions
})

submission.to_csv("submission.csv",index=False)

print(submission.head())

'''

## **Model1 - LSTM_Baseline**

In [ ]:
'''
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df_train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

'''

In [ ]:
'''
dups=df_train[df_train.duplicated(subset='prompt',keep='first')]
print(len(dups))
'''

In [ ]:
'''
df_dup = df_train.drop_duplicates(subset='prompt', keep='first').reset_index(drop=True)
print("Original:", len(df_train))       
print("After dup:", len(df_dup))  
'''

In [ ]:
'''
train_data, val_data = train_test_split(
    df_dup,
    test_size=0.2,
    stratify=df_dup['answer'],
    random_state=42
)

print("Train size:", len(train_data))
print("Val size:", len(val_data))
'''

In [ ]:

'''
all_text=[]
for _,row in train_data.iterrows():
    all_text.append(row['prompt'])
    all_text.append(row['A'])
    all_text.append(row['B'])
    all_text.append(row['C'])
    all_text.append(row['D'])
    all_text.append(row['E'])
    
'''

In [ ]:
'''
import re
clean_text = []

for sentence in all_text:
    sentence = re.sub(r"[^\w\s]"," ",sentence.lower())
    clean_text.append(sentence)

'''

In [ ]:
'''
from collections import Counter

text = " ".join(clean_text)

word_counts = Counter(text.split())
'''

In [ ]:
#print(f'Total word length after splitting each words from the sentences : {len(word_counts)}')

In [ ]:
# filtering out the rare words , rare ones is being replaced by UNK and PAD
'''
vocab={'<PAD>': 0, '<UNK>': 1}

for word,counts in word_counts.items():
    if counts >= 3:
        vocab[word] = len(vocab)

print(len(vocab))
'''

In [ ]:
# Encoding

'''
def encode(text,vocab,max_len=128):
    text = re.sub(r"[^\w\s]", " ", text.lower())
    words = text.split()
    ids=[]
    for word in words:
        if word in vocab:
            ids.append(vocab[word])
        else:
            ids.append(vocab['<UNK>'])
    ids = ids[:max_len]
    ids =ids+[0]*(max_len - len(ids))
    return ids
'''

In [ ]:
'''
test_sentence = "What is quantum entanglement stuff"
encoded = encode(test_sentence, vocab)
print(len(encoded))   
print(encoded[:10])   
'''

In [ ]:
'''
from torch.utils.data import Dataset


class MCQ(Dataset):
    def __init__(self,df,vocab):
        self.df = df 
        self.vocab = vocab 

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):
        row = self.df.iloc[idx]

        inputs=[]

        for option in ["A","B","C","D","E"]:
            text = row["prompt"] + " " + row[option]
            inputs.append(encode(text,self.vocab))
            
        label_map = {'A':0, 'B':1, 'C':2, 'D':3, 'E':4}
        label = label_map[row['answer']]   
    
        return torch.tensor(inputs), torch.tensor(label)
'''

In [ ]:
'''
train_dataset = MCQ(train_data,vocab)
val_dataset = MCQ(val_data,vocab)
'''

In [ ]:
'''
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

'''

In [ ]:
#inputs, labels = next(iter(train_loader))

In [ ]:
#inputs.shape

In [ ]:
#inputs[0][0]

In [ ]:
#labels

In [ ]:
#labels.shape

In [ ]:
#len(vocab)

### ***LSTM model*** ###

In [ ]:
'''
from torch.nn.utils.rnn import pack_padded_sequence

class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embd_dim, hiddn_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embd_dim, padding_idx=0) 
        self.lstm = nn.LSTM(embd_dim, hiddn_dim, batch_first=True)
        self.fc   = nn.Linear(hiddn_dim, 1)

    def forward(self, x):
        scores = []
        for i in range(5):
            option = x[:, i, :]
            lengths = (option != 0).sum(dim=1).cpu().clamp(min=1) 
            embedded = self.embedding(option)
            packed = pack_padded_sequence(embedded, lengths, batch_first=True, enforce_sorted=False)  
            output, (hidden, cell) = self.lstm(packed)
            score = self.fc(hidden.squeeze(0))
            scores.append(score)
        return torch.cat(scores, dim=1)


'''

In [ ]:
'''
model = LSTMModel(
    vocab_size=len(vocab),
    embd_dim=128,
    hiddn_dim=256
)

print(model)

'''

In [ ]:
'''

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

'''

In [ ]:
'''
wandb.init(project="mcq-solver", name="lstm-baseline")

for epoch in range(10):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    model.eval()
    correct = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
    
    acc = correct / len(val_dataset)
    print(f"Epoch {epoch+1} | Loss: {total_loss:.3f} | Val Acc: {acc:.3f}")
    
 
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": total_loss,
        "val_accuracy": acc
    })
    
     

wandb.finish()

'''

In [ ]:
#print(test_df.columns.tolist())

In [ ]:
'''
def predict(model,test_df,vocab):
    model.eval()
    predictions=[]
    label_map={0:'A', 1:'B', 2:'C', 3:'D', 4:'E'}


    for _,row in test_df.iterrows():
        if pd.notna(row['known_answer']):
            
            known = row['known_answer']
            freq = [x for x in ['B','C','A','D','E'] if x!=known]
            pred = known + " " + freq[0]+ " "+freq[1]

        else:

            inputs=[]

            for opt in  ['A','B','C','D','E']:
                text = row['prompt'] + " " + row[opt]
                inputs.append(encode(text,vocab))


            inputs_tensor = torch.tensor([inputs])
            with torch.no_grad():
                scores = model(inputs_tensor)
                ranked = torch.argsort(scores, dim=1, descending=True)[0]
                
                
                top3 = [label_map[r.item()] for r in ranked[:3]]
                pred = " ".join(top3)
        predictions.append(pred)
        
    return predictions

'''

In [ ]:
'''
predictions = predict(model, test_df, vocab)

submission = pd.DataFrame({
    'ID': test_df['id'],
    'Prediction': predictions
})

submission.to_csv('submission.csv', index=False)
print(submission.head(10))
print("Total predictions:", len(predictions))

'''

# MODEL 2 - BERT


In [ ]:
import pandas as pd

import torch

print(torch.cuda.memory_allocated()/1024**3)
print(torch.cuda.memory_reserved()/1024**3)

In [ ]:
df_train=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

df_test=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

In [ ]:
df_train.columns

In [ ]:
from transformers import BertTokenizer,BertForMultipleChoice



model = BertForMultipleChoice.from_pretrained("bert-base-uncased")




print("Before:", torch.cuda.memory_allocated()/1024**3)

model.to("cuda")

print("After:", torch.cuda.memory_allocated()/1024**3)

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
from sklearn.model_selection import train_test_split


In [ ]:
train_data, val_data = train_test_split(df_train,test_size=0.2,stratify=df_train['answer'],random_state=42)

print("Train size:", len(train_data))
print("Val size:", len(val_data))

In [ ]:
def map3(predictions,labels):
    score=0.0
    for pred,label in zip(predictions,labels):
        pred=pred.tolist()
        label=label.item()

        if label in pred:
            rank = pred.index(label)+1
            score += 1.0/rank
    return score

In [ ]:
from torch.utils.data import Dataset,DataLoader

import torch

In [ ]:

class MCQ_Bert(Dataset):
    def __init__(self,df,tokenizer,max_length=128):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length=max_length
        
        

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):
        row = self.df.iloc[idx]

        questions = [row['prompt']]*5

        choices = [row["A"],row["B"],row["C"],row["D"],row["E"]]

        encoding = self.tokenizer(questions,choices,padding="max_length",truncation=True,max_length=self.max_length,return_tensors="pt")

            
        label_map = {'A':0, 'B':1, 'C':2, 'D':3, 'E':4}
        labels = label_map[row['answer']]   
    
        return {"input_ids":encoding["input_ids"].squeeze(0),"attention_mask":encoding["attention_mask"].squeeze(0),"labels":torch.tensor(labels)}


In [ ]:
train_dataset = MCQ_Bert(train_data,tokenizer)
val_dataset = MCQ_Bert(val_data,tokenizer)



In [ ]:

train_loader = DataLoader(train_dataset,batch_size=8,shuffle=True)

val_loader = DataLoader(val_dataset,batch_size=8,shuffle=False)

In [ ]:


optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [ ]:
#wandb.init(project="mcq-solver", name="bert")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model.to(device)

for epoch in range(3):
    
    model.train()

    total_loss=0

    for batch in train_loader:
        
        # moving to gpu
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad() #removing old grads

        #forw pass

        outputs = model(input_ids=input_ids,attention_mask=attention_mask,labels=labels)

        loss = outputs.loss

        total_loss += loss.item()

        #compute gradients
        loss.backward()

        optimizer.step() # upd model weights

        avg_train_loss = total_loss/len(train_loader)

     

In [ ]:
model.eval()

total_score = 0
total_samples = 0

with torch.no_grad():

    for batch in val_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids,attention_mask=attention_mask)
        top3 = torch.topk(outputs.logits,k=3,dim=1).indices
        total_score += map3(top3,labels)
        total_samples += labels.size(0)



map3_score = total_score/total_samples

print(f"Epoch {epoch+1}")
print(f"Train Loss : {avg_train_loss:.4f}")
print(f"Val MAP@3 : {map3_score:.4f}")


     

#wandb.log({"epoch": epoch + 1,"train_loss": avg_train_loss,"val_map3": map3_score })

#wandb.finish()


In [ ]:
# test dataset


class MCQ_BertTest(Dataset):
    def __init__(self,df,tokenizer,max_length=128):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length=max_length
        
        

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):
        row = self.df.iloc[idx]

        questions = [row['prompt']]*5

        choices = [row["A"],row["B"],row["C"],row["D"],row["E"]]

        encoding = self.tokenizer(questions,choices,padding="max_length",truncation=True,max_length=self.max_length,return_tensors="pt")
    
        return {"input_ids":encoding["input_ids"].squeeze(0),"attention_mask":encoding["attention_mask"].squeeze(0)}



In [ ]:
test_dataset =  MCQ_BertTest(df_test, tokenizer)

In [ ]:
test_loader = DataLoader(test_dataset,batch_size=8,shuffle=False)

In [ ]:
model.eval()

predictions = []
with torch.no_grad():
    
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids,attention_mask=attention_mask)
        top3 = torch.topk(outputs.logits, k=3, dim=1).indices
        index = {0: "A",1: "B",2: "C",3: "D",4: "E"}  
        for pred in top3:
            answer=""
            for i in pred:
                 answer = answer + index[i.item()]+" "
            predictions.append(answer.strip())

In [ ]:

len(predictions)

In [ ]:
submission = pd.DataFrame({"ID": df_test["id"],"Prediction":predictions})

In [ ]:
submission.to_csv("submission.csv", index=False)